In [1]:
import sys
from pathlib import Path
import json
from datasets import load_dataset
import torch
import torch.nn as nn
from torchvision.transforms import transforms

In [2]:
# Thư mục gốc của project
ROOT_DIR = Path.cwd().parent
ROOT_DIR

WindowsPath('D:/Code Module 6/VietNamese_Image_Captioning_with_CNN_LTSM')

In [3]:
dataset = load_dataset("ai-enthusiasm-community/KTVIC")

In [4]:
dataset['train'][0]

{'image_uid': '000000000001',
 'caption_uid': ['000000000001_1',
  '000000000001_2',
  '000000000001_3',
  '000000000001_4',
  '000000000001_5'],
 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=695x456>,
 'caption_vi': ['ba chiếc thuyền đang di chuyển ở trên con sông',
  'có ba con thuyền đang di chuyển trên con sông',
  'trên dòng sông có ba con thuyền đang di chuyển',
  'ba con thuyền đang di chuyển bên một cánh đồng lúa',
  'ba chiếc thuyền đang chuyển động trên một con sông'],
 'segment_caption_vi': ['ba chiếc thuyền đang di_chuyển ở trên con sông',
  'có ba con thuyền đang di_chuyển trên con sông',
  'trên dòng sông có ba con thuyền đang di_chuyển',
  'ba con thuyền đang di_chuyển bên một cánh đồng lúa',
  'ba chiếc thuyền đang chuyển_động trên một con sông']}

In [5]:
SOURCE_DIR = ROOT_DIR/'source'
MODEL_DIR = ROOT_DIR/'model'
CONFIG_DIR = ROOT_DIR/'configuration'

In [6]:
# Thêm các thư mục vào không gian tìm kiếm
sys.path.extend([str(ROOT_DIR), str(SOURCE_DIR), str(MODEL_DIR), str(CONFIG_DIR)])

# Load vocab is built in file.json

In [7]:
from vocabulary.vocab import *

In [8]:
tokenizer = Tokenizer()

In [9]:
PATH_SAVE_VOCAB = CONFIG_DIR/'vocab.json'

In [10]:
vocab = BuiltVocabFromIterator()

In [11]:
vocab.load(path=str(PATH_SAVE_VOCAB))

In [12]:
vocab_size = vocab.vocab_size
vocab_size

600

In [13]:
vocab.stoi()

{'<padd>': 0,
 '<start>': 1,
 '<end>': 2,
 '<unk>': 3,
 'có': 4,
 'một': 5,
 'ở': 6,
 'đang': 7,
 'người': 8,
 'hiện': 9,
 'xuất': 10,
 'trên': 11,
 'của': 12,
 'hàng': 13,
 'bên': 14,
 'nhiều': 15,
 'trong': 16,
 'chiếc': 17,
 'những': 18,
 'trước': 19,
 'nữ': 20,
 'áo': 21,
 'màu': 22,
 'phụ': 23,
 'mặc': 24,
 'được': 25,
 'đứng': 26,
 'hai': 27,
 'xe': 28,
 'đường': 29,
 'phía': 30,
 'gái': 31,
 'cô': 32,
 'sự': 33,
 'nhà': 34,
 'khu': 35,
 'đàn': 36,
 'ông': 37,
 'bức': 38,
 'cửa': 39,
 'là': 40,
 'ảnh': 41,
 'cái': 42,
 'quầy': 43,
 'con': 44,
 'cây': 45,
 'vài': 46,
 'mặt': 47,
 'bày': 48,
 'di': 49,
 'bán': 50,
 'tay': 51,
 'hình': 52,
 'xanh': 53,
 'ngồi': 54,
 'máy': 55,
 'đây': 56,
 'ngôi': 57,
 'và': 58,
 'chuyển': 59,
 'khung': 60,
 'trắng': 61,
 'chợ': 62,
 'đi': 63,
 'chụp': 64,
 'cảnh': 65,
 'thị': 66,
 'siêu': 67,
 'các': 68,
 'đồ': 69,
 'này': 70,
 'đỏ': 71,
 'ăn': 72,
 'cầm': 73,
 'kệ': 74,
 'phố': 75,
 'vào': 76,
 'đen': 77,
 'nhóm': 78,
 'căn': 79,
 'nhau': 80,
 'lá

In [14]:
vocab.itos()

{'0': '<padd>',
 '1': '<start>',
 '2': '<end>',
 '3': '<unk>',
 '4': 'có',
 '5': 'một',
 '6': 'ở',
 '7': 'đang',
 '8': 'người',
 '9': 'hiện',
 '10': 'xuất',
 '11': 'trên',
 '12': 'của',
 '13': 'hàng',
 '14': 'bên',
 '15': 'nhiều',
 '16': 'trong',
 '17': 'chiếc',
 '18': 'những',
 '19': 'trước',
 '20': 'nữ',
 '21': 'áo',
 '22': 'màu',
 '23': 'phụ',
 '24': 'mặc',
 '25': 'được',
 '26': 'đứng',
 '27': 'hai',
 '28': 'xe',
 '29': 'đường',
 '30': 'phía',
 '31': 'gái',
 '32': 'cô',
 '33': 'sự',
 '34': 'nhà',
 '35': 'khu',
 '36': 'đàn',
 '37': 'ông',
 '38': 'bức',
 '39': 'cửa',
 '40': 'là',
 '41': 'ảnh',
 '42': 'cái',
 '43': 'quầy',
 '44': 'con',
 '45': 'cây',
 '46': 'vài',
 '47': 'mặt',
 '48': 'bày',
 '49': 'di',
 '50': 'bán',
 '51': 'tay',
 '52': 'hình',
 '53': 'xanh',
 '54': 'ngồi',
 '55': 'máy',
 '56': 'đây',
 '57': 'ngôi',
 '58': 'và',
 '59': 'chuyển',
 '60': 'khung',
 '61': 'trắng',
 '62': 'chợ',
 '63': 'đi',
 '64': 'chụp',
 '65': 'cảnh',
 '66': 'thị',
 '67': 'siêu',
 '68': 'các',
 '69': '

In [15]:
vocab.default_key

'<unk>'

# Lấy thông số của mean và std để normalize ảnh đã được lưu trong file.json

In [16]:
from preprocessing.preprocessing_img import*

In [17]:
computer = ComputeMeanStd()

In [18]:
mean_std_path = CONFIG_DIR/'mean_std_img.json'

In [19]:
mean, std = computer.load_mean_std(str(mean_std_path))

In [20]:
print(mean)
print(std)

tensor([0.5008, 0.4719, 0.4341])
tensor([0.2770, 0.2680, 0.2809])


# Khởi tạo kiểu DL vicaptionDatset đã được xây dựng

In [21]:
from organize_data import * 

In [22]:
vicaptiondataset = VicaptioningDataSet(dataset= dataset, split='train', val_split=0.2, is_shuffle=True, transform=[transforms.Normalize(mean= mean, std=std)], max_length=20, vocab=vocab)

In [23]:
vicaption_valdataset =VicaptioningDataSet(dataset= dataset, split='valid', val_split=0.2, is_shuffle=True, transform=[transforms.Normalize(mean= mean, std=std)], max_length=20, vocab=vocab)

In [24]:
X, y = vicaptiondataset[0]

In [25]:
X.shape

torch.Size([3, 224, 224])

In [26]:
y.shape

torch.Size([20])

# Khởi tạo mô hình và huấn luyện mô hình ViCaptioningImgModel

In [27]:
vocab.vocab_size

600

In [28]:
from build_model import *

In [29]:
ModelGenerateCaption = ViCaptioningImgModel(version_ResNet=34, vocab_size=vocab.vocab_size, embedding_dim=100,hidden_size=100,num_layer=1)

In [30]:
Trainer = TrainModel()

In [31]:
criterion = nn.CrossEntropyLoss()

In [32]:
optimizer = torch.optim.Adam(ModelGenerateCaption.parameters(), lr=0.001)

In [33]:
Trainer.fit(model= ModelGenerateCaption, 
            batch_size=128, 
            criterion=criterion, 
            optimizer=optimizer, 
            dataset=vicaptiondataset, 
            is_shuffle=True, 
            n_epochs=7,val_dataset=vicaption_valdataset)

 14%|███████████▊                                                                       | 1/7 [03:13<19:22, 193.83s/it]

Epoch [   1/ 7]  - Loss = 2.8475 - Accuracy = 0.6201 - Loss_Validation = 1.0785 - Accuracy = 0.8697


 29%|███████████████████████▋                                                           | 2/7 [06:45<17:00, 204.07s/it]

Epoch [   2/ 7]  - Loss = 0.5984 - Accuracy = 0.9328 - Loss_Validation = 0.2958 - Accuracy = 0.9696


 43%|███████████████████████████████████▌                                               | 3/7 [10:10<13:38, 204.64s/it]

...

Epoch [   5/ 7]  - Loss = 0.0360 - Accuracy = 0.9998 - Loss_Validation = 0.0250 - Accuracy = 1.0000


 86%|███████████████████████████████████████████████████████████████████████▏           | 6/7 [20:52<03:32, 212.82s/it]

Epoch [   6/ 7]  - Loss = 0.0190 - Accuracy = 1.0000 - Loss_Validation = 0.0145 - Accuracy = 1.0000


100%|███████████████████████████████████████████████████████████████████████████████████| 7/7 [24:27<00:00, 209.63s/it]

Epoch [   7/ 7]  - Loss = 0.0118 - Accuracy = 1.0000 - Loss_Validation = 0.0098 - Accuracy = 1.0000


In [37]:
PATH_SAVE_WEIGHT = ROOT_DIR/'model_weight.pth'

In [41]:
# Lưu trọng số đã huấn luyện của mô hình 
torch.save(ModelGenerateCaption.state_dict(),PATH_SAVE_WEIGHT)